# Multiple qubits

So far we have worked with a single qubit, building up three representations:

| Representation | Qubit state | Unitary transform |
|---|---|---|
| Schrödinger | State vector | 2×2 unitary matrix |
| Heisenberg — observable matrices | Observable matrices $\sigma_x$, $\sigma_z$ | Matrix sandwich $U^\dagger\,O\,U$ |
| Heisenberg — Pauli expressions | Pauli expressions | Pauli expression sandwich |

In the Schrödinger picture a qubit's state is a 2-element vector and every gate is a 2×2 unitary matrix that acts on it. In the Heisenberg picture the state is instead encoded in the qubit's observables — the expressions that represent what a measurement would return — and gates transform those observables via the sandwich $U^\dagger\,O\,U$, which can be evaluated either using matrices or Pauli algebra.

For the Pauli expression representation we have also written Python code in `gates_single.py` that shortcuts the SymPy sandwich evaluation: rather than computing $U^\dagger\,O\,U$ symbolically, each gate function directly applies the known transformation to the qubit's observables. Unit tests verify that this shortcut produces the same result as evaluating the full Pauli expression sandwich via `qubit.evolve`.

We now go through the same progression for a **two-qubit system**.

## Schrödinger state of two qubits

The combined Schrödinger state of two qubits is the **tensor product** of their individual state vectors. If the first qubit is in state $\ket{\psi_1}$ and the second in state $\ket{\psi_2}$, the joint state is:

$$\ket{\psi_1} \otimes \ket{\psi_2}$$

The tensor product of a 2-element vector with another 2-element vector produces a 4-element vector: each entry of the first vector is multiplied by the entire second vector, and the results are stacked.

**Example 1: both qubits in $\ket{0}$**

$$\ket{0} \otimes \ket{0}
= \begin{bmatrix}1\\0\end{bmatrix} \otimes \begin{bmatrix}1\\0\end{bmatrix}
= \begin{bmatrix}1\cdot\begin{bmatrix}1\\0\end{bmatrix}\\[4pt]0\cdot\begin{bmatrix}1\\0\end{bmatrix}\end{bmatrix}
= \begin{bmatrix}1\\0\\0\\0\end{bmatrix}$$

**Example 2: first qubit in $\ket{+}$, second qubit in $\ket{0}$**

$$\ket{+} \otimes \ket{0}
= \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix} \otimes \begin{bmatrix}1\\0\end{bmatrix}
= \frac{1}{\sqrt{2}}\begin{bmatrix}1\cdot\begin{bmatrix}1\\0\end{bmatrix}\\[4pt]1\cdot\begin{bmatrix}1\\0\end{bmatrix}\end{bmatrix}
= \frac{1}{\sqrt{2}}\begin{bmatrix}1\\0\\1\\0\end{bmatrix}$$

Combining two qubits into a joint state by taking the tensor product is straightforward. Going in the other direction — recovering the individual qubit states from a 4-element vector — is harder. In general, a 4-element state vector cannot always be factored back into a product of two independent 2-element vectors. When factoring is impossible the qubits are **entangled**: the outcome of measuring one qubit is correlated with the outcome of measuring the other, and neither qubit has a well-defined state on its own. We will see a lot more of this when we get to quantum teleportation.

For now it is worth noting that it is not at all obvious from inspecting a state vector whether the qubits are entangled or entirely separable — the two examples above both look like plain lists of numbers, yet one may be factorable and another may not.

## Applying a gate to one qubit of a two-qubit system

To apply a gate to only one qubit in a two-qubit system, the gate must be extended into a 4×4 matrix that acts on the joint state vector. For a gate $G$ acting on the first qubit while leaving the second unchanged, this is done by taking the tensor product of $G$ with the 2×2 identity matrix $I$:

$$(G \otimes I)\,\ket{\psi_1\psi_2}$$

**Setup: both qubits initialised to $\ket{0}$**

$$\ket{00} = \ket{0} \otimes \ket{0} = \begin{bmatrix}1\\0\\0\\0\end{bmatrix}$$

**Applying Hadamard to the first qubit**

The Hadamard matrix and the identity matrix are:

$$H = \frac{1}{\sqrt{2}}\begin{bmatrix}1&1\\1&-1\end{bmatrix} \qquad I = \begin{bmatrix}1&0\\0&1\end{bmatrix}$$

Their tensor product replaces each entry of $H$ with that scalar multiple of $I$:

$$H \otimes I = \frac{1}{\sqrt{2}}\begin{bmatrix}1\cdot\begin{bmatrix}1&0\\0&1\end{bmatrix} & 1\cdot\begin{bmatrix}1&0\\0&1\end{bmatrix} \\[6pt] 1\cdot\begin{bmatrix}1&0\\0&1\end{bmatrix} & {-1}\cdot\begin{bmatrix}1&0\\0&1\end{bmatrix}\end{bmatrix} = \frac{1}{\sqrt{2}}\begin{bmatrix}1&0&1&0\\0&1&0&1\\1&0&-1&0\\0&1&0&-1\end{bmatrix}$$

Applying this to $\ket{00}$:

$$(H \otimes I)\,\ket{00} = \frac{1}{\sqrt{2}}\begin{bmatrix}1&0&1&0\\0&1&0&1\\1&0&-1&0\\0&1&0&-1\end{bmatrix}\begin{bmatrix}1\\0\\0\\0\end{bmatrix} = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\0\\1\\0\end{bmatrix}$$

The result $\frac{1}{\sqrt{2}}[1,0,1,0]^\top$ is exactly $\ket{+} \otimes \ket{0}$, confirming that the Hadamard transformed the first qubit from $\ket{0}$ to $\ket{+}$ while the second qubit remained in $\ket{0}$.

**The density problem**

This representation is dense. Even though the second qubit was completely unaffected by the gate, we still had to construct a 4×4 matrix to express the operation. A three-qubit system would require an 8×8 matrix, and in general an $n$-qubit system requires a $2^n \times 2^n$ matrix. At 10 qubits that is a $1024 \times 1024$ matrix — unwieldy to write down or reason about even when most of the gate's entries are just copying the identity through.

Symbolic algebra systems like SymPy have a sparse matrix representation, where only the non-zero entries are stored and the full matrix is reconstructed only when needed. This helps with storage, but the representation remains hard to read and reason about intuitively.

When we move on to the Pauli expression representation we will see that it is naturally more compact: a gate on one qubit of a ten-qubit system is still written as a short Pauli expression, with no need to expand it into a large matrix.

## The CNOT gate

The **CNOT** (controlled-NOT) gate is a two-qubit gate. It leaves the first qubit (the *control*) unchanged, and flips the second qubit (the *target*) if and only if the control is in state $\ket{1}$:

$$\ket{00} \to \ket{00} \qquad \ket{01} \to \ket{01} \qquad \ket{10} \to \ket{11} \qquad \ket{11} \to \ket{10}$$

As a 4×4 matrix acting on the joint state vector, this is:

$$\text{CNOT} = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{bmatrix}$$

The top-left 2×2 block is the identity (control is $\ket{0}$, target is left alone); the bottom-right 2×2 block is the Pauli X matrix (control is $\ket{1}$, target is flipped).

**Applying CNOT to $\ket{00}$**

$$\text{CNOT}\,\ket{00} = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{bmatrix}\begin{bmatrix}1\\0\\0\\0\end{bmatrix} = \begin{bmatrix}1\\0\\0\\0\end{bmatrix} = \ket{00}$$

The state is unchanged. Because the control qubit is $\ket{0}$, the gate has nothing to do and the target qubit is left in $\ket{0}$.

**Applying CNOT to $\ket{10}$**

$$\text{CNOT}\,\ket{10} = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{bmatrix}\begin{bmatrix}0\\0\\1\\0\end{bmatrix} = \begin{bmatrix}0\\0\\0\\1\end{bmatrix} = \ket{11}$$

This time the control qubit is $\ket{1}$, so the target qubit is flipped from $\ket{0}$ to $\ket{1}$.

## CNOT in the Pauli observable representation

In the Pauli representation, the CNOT gate is expressed as a Pauli expression rather than a 4×4 matrix. It is derived by decomposing by the control qubit's eigenstates (as shown in `gates_multi.py`):

$$\text{CNOT} = \tfrac{1}{2}(1 + \sigma_z^{(c)} + \sigma_x^{(t)} - \sigma_z^{(c)}\,\sigma_x^{(t)})$$

The superscripts $(c)$ and $(t)$ are the qubit labels — they indicate which qubit each Pauli symbol belongs to. In the code these are just strings passed as the `label` argument. Because $\sigma_z^{(c)}$ and $\sigma_x^{(t)}$ act on different qubits they commute, so their product is simply a symbolic juxtaposition rather than a Pauli algebra reduction.

In SymPy this expression is written as:

```python
½ * (1 + Pauli(3, label="c") + Pauli(1, label="t") - Pauli(3, label="c") * Pauli(1, label="t"))
```

Rather than evaluating the full sandwich $U^\dagger O U$ for each observable, `gates_multi.cnot` directly applies the known observable transformations as a shortcut. Unit tests in `tests/test_gates_multi.py` verify that this shortcut produces identical results to using `qubit.evolve` with the Pauli expression above.

In [1]:
from qubit import Qubit
from gates_multi import cnot
from IPython.display import display

c = Qubit.qubit_time_0("c")
t = Qubit.qubit_time_0("t")

c_after, t_after = cnot(c, t)
display(c_after)
display(t_after)

Qubit(c, c1*t1, c2*t1, c3)

Qubit(t, t1, c3*t2, c3*t3)

The results show that after CNOT:

- The **target** qubit's Z observable has become $\sigma_z^{(c)}\,\sigma_z^{(t)}$ — entangled with the control qubit's Z observable. Its X observable is unchanged.
- The **control** qubit's Z observable is completely unaffected, remaining $\sigma_z^{(c)}$. However, its X observable has become $\sigma_x^{(c)}\,\sigma_x^{(t)}$ — entangled with the X observable of the target qubit.

This gate is somewhat symmetrical: both qubits affect each other.

However, because we usually think about qubits in the computational basis — which corresponds to the Z observable — it is easy to slip into thinking of CNOT as a purely classical gate, where the control is unchanged and the target is conditionally flipped. In a classical CNOT that really is the full story.

In quantum computing there is no such one-way effect. Entanglement has a measurable influence on both qubits; it is just that to see the effect on the control qubit you have to measure it in the Hadamard basis rather than the computational basis — or equivalently, apply a Hadamard transform to the control qubit before measuring it in the Z basis. Only then does the entanglement in the X observable become visible.

## CNOT in the Hadamard basis

To make the symmetry concrete, we can pass $\ket{+-}$ through the CNOT gate in the matrix representation and see what comes out.

**Setting up $\ket{+-}$**

$$\ket{+} \otimes \ket{-}
= \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix} \otimes \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}
= \frac{1}{2}\begin{bmatrix}1\cdot\begin{bmatrix}1\\-1\end{bmatrix}\\[4pt]1\cdot\begin{bmatrix}1\\-1\end{bmatrix}\end{bmatrix}
= \frac{1}{2}\begin{bmatrix}1\\-1\\1\\-1\end{bmatrix}$$

**Applying CNOT**

$$\text{CNOT}\,\ket{+-} = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{bmatrix}\frac{1}{2}\begin{bmatrix}1\\-1\\1\\-1\end{bmatrix} = \frac{1}{2}\begin{bmatrix}1\\-1\\-1\\1\end{bmatrix}$$

**Identifying the result as $\ket{--}$**

$$\ket{-} \otimes \ket{-}
= \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix} \otimes \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}
= \frac{1}{2}\begin{bmatrix}1\\-1\\-1\\1\end{bmatrix}$$

The result is exactly $\ket{--}$.

In the Hadamard basis, $\ket{-}$ plays the same role as $\ket{1}$ in the computational basis — it is the $-1$ eigenstate of the X observable. Looked at through that lens, the second qubit started in $\ket{-}$ and ended in $\ket{-}$: it was unchanged. The first qubit started in $\ket{+}$ (the $+1$ eigenstate, playing the role of $\ket{0}$) and ended in $\ket{-}$ (the $-1$ eigenstate, playing the role of $\ket{1}$): it was flipped.

In the Hadamard basis the roles of control and target are reversed. The second qubit — which looks like the target in the computational basis — is unchanged, and the first qubit — which looks like the control — is flipped according to the value of the second. This is not a coincidence or a trick; it is the same symmetry we saw in the Pauli observable representation, where the control's X observable became entangled with the target's X observable. The Hadamard basis is simply the basis in which that entanglement becomes directly visible.

Notice that in the matrix approach we had to work this out by choosing a specific input state, applying the gate, and then recognising the output. The behaviour had to be discovered. In the Pauli observable representation, it was already written down plainly in the result: the control qubit's X observable became $\sigma_x^{(c)}\,\sigma_x^{(t)}$, directly telling us that the control's X measurement is affected by the target's X observable. No special input state or change of basis was needed to see it.

## Y-rotation followed by CNOT

To see something more interesting, we can rotate the control qubit by an angle $\theta$ around the Y axis before applying CNOT. This puts the control into a superposition that depends on $\theta$, and the CNOT then entangles that superposition with the target.

In [2]:
from qubit import Qubit
from gates_single import yrotation
from gates_multi import cnot
from sympy import Symbol
from IPython.display import display

theta = Symbol('theta', real=True)

c = Qubit.qubit_time_0('c')
t = Qubit.qubit_time_0('t')

c_rotated = yrotation(c, theta)
c_after, t_after = cnot(c_rotated, t)

display(c_after)
display(t_after)

Qubit(c, (sin(theta)*c3 + cos(theta)*c1)*t1, c2*t1, -sin(theta)*c1 + cos(theta)*c3)

Qubit(t, t1, (-sin(theta)*c1 + cos(theta)*c3)*t2, (-sin(theta)*c1 + cos(theta)*c3)*t3)

The control qubit's X observable after the Y-rotation is $\cos\theta\,\sigma_x^{(c)} + \sin\theta\,\sigma_z^{(c)}$, a blend of X and Z depending on the rotation angle. After CNOT that entire expression becomes entangled with the target's X observable:

$$X_c \;\to\; (\cos\theta\,\sigma_x^{(c)} + \sin\theta\,\sigma_z^{(c)})\,\sigma_x^{(t)}$$

The target qubit's Z observable has been similarly entangled with the control's Z observable, which the Y-rotation has itself mixed with the control's original X observable:

$$Z_t \;\to\; (\cos\theta\,\sigma_z^{(c)} - \sin\theta\,\sigma_x^{(c)})\,\sigma_z^{(t)}$$

At $\theta = 0$ no rotation is applied and we recover the standard CNOT result. At $\theta = \pi/2$ the control's Z observable plays the role its X observable normally does, and the entanglement structure rotates accordingly. The Pauli expression representation makes this $\theta$-dependence explicit and readable directly from the output, with no need to work through matrix multiplications or choose test input states.